In [1]:
from Auxiliaries import evaluate_policy
from Policies import ProBSP
from NewEnvironment import Inventory, GeometricOrderPipeline, InventoryRS
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
from sb3_contrib import MaskablePPO as PPO
from MaskedDQN import MaskedDoubleDQN as DDQN

This file provides an example on defining an environment and learning a DQN, and PPO policies.
The inventory problem has the following parameters:
M=2, p=0.33, B=3, MTTF=10, Co=2, Ce=5


In [2]:
num_machines = 2
lead_times_p = 1/3
max_batch_size = 3
mttf = 10
sort_degradation = True  # Sort degradation to benefit from the reduction in state space

order_pipeline = GeometricOrderPipeline(num_machines, lead_times_p)
inventory = Inventory(machines=num_machines,
                      order_pipeline=order_pipeline,
                      mttf=mttf,
                      sorted_degradation=sort_degradation)

n, xo = 0, 60
probsp = ProBSP(env=inventory, n=n, xo=xo, max_batch_size=max_batch_size)

# Evaluate the ProBSP

In [3]:
evaluate_policy(env=inventory, policy=probsp, replication=8, processors=4)

ProBSP with N=0 and Xo=60.0 - Inventory - 8 replications - 20000 steps - 2000 burn-in period avg:
Costs=0.9312644879778192 ± 0.0086 	 E[S]=0.1881 ± 0.0036 	 FR=0.6021 ± 0.0087 	 Total Eval Time=3.3916


(np.float64(0.9312644879778192),
 np.float64(0.008624203741236805),
 np.float64(0.6021242107558884),
 np.float64(0.008737436587872136),
 np.float64(0.18809659090909064),
 np.float64(0.0035630708546543813))

# Let us learn using a PPO policy

In [4]:
print("Learning a policy using PPO")
ppo = PPO(MaskableActorCriticPolicy, env=inventory, verbose=0)
ppo.learn(200000)
print("Finished Learning PPO policy, evaluating .......")
evaluate_policy(env=inventory, policy=ppo, replication=8, processors=4)

Learning a policy using PPO
Finished Learning PPO policy, evaluating .......
<sb3_contrib.ppo_mask.ppo_mask.MaskablePPO object at 0x148a825c0> - Inventory - 8 replications - 20000 steps - 2000 burn-in period avg:
Costs=0.8956126994227536 ± 0.0066 	 E[S]=0.2672 ± 0.0038 	 FR=0.5659 ± 0.0059 	 Total Eval Time=40.2762


(np.float64(0.8956126994227536),
 np.float64(0.006644848491569335),
 np.float64(0.5658621899590253),
 np.float64(0.005873373568115137),
 np.float64(0.2671534090909088),
 np.float64(0.003790615961798094))

# Let us learn using a DQN policy

In [14]:
print("Learning a policy using DQN")
dqn = DDQN(env=inventory)
ppo.learn(800000)
print("Finished Learning DQN policy, evaluating .......")
evaluate_policy(env=inventory, policy=dqn, replication=8, processors=4)

Learning a policy using DQN
Finished Learning DQN policy, evaluating .......
DDQN - Inventory - 8 replications - 20000 steps - 2000 burn-in period avg:
Costs=1.331967865097041 ± 0.0056 	 E[S]=0.0000 ± 0.0000 	 FR=0.0002 ± 0.0000 	 Total Eval Time=16.1519


(np.float64(1.331967865097041),
 np.float64(0.005581406124059565),
 np.float64(0.00023881729258758144),
 np.float64(1.0010444217523183e-06),
 np.float64(0.0),
 np.float64(0.0))

# Inventory with Reward Shaping using the ProBSP

In [8]:
# When reward shaping is included using the ProBSP
inventory_rs = InventoryRS(machines=num_machines,
                           order_pipeline=order_pipeline,
		                   mttf=mttf,
		                   sorted_degradation=sort_degradation,
                           probsp=True,
                           bsp=False,
                           probsp_xo=xo,
                           probsp_n=n,
                           gamma=0.0001
                           )

In [ ]:
# Again, let us learn using a PPO policy
print("Learning a policy using PPO")
ppo = PPO(MaskableActorCriticPolicy, env=inventory_rs, verbose=0)
ppo.learn(200000)
print("Finished Learning PPO policy, evaluating .......")
evaluate_policy(env=inventory_rs, policy=ppo, replication=8, processors=4)


Learning a policy using PPO
Finished Learning PPO policy, evaluating .......
<sb3_contrib.ppo_mask.ppo_mask.MaskablePPO object at 0x14c096fb0> - Inventory-RS - 8 replications - 20000 steps - 2000 burn-in period avg:
Costs=0.8846018362801691 ± 0.0073 	 E[S]=0.2708 ± 0.0057 	 FR=0.5871 ± 0.0053 	 Total Eval Time=41.9841
Learning a policy using DQN
Finished Learning DQN policy, evaluating .......
DDQN - Inventory-RS - 8 replications - 20000 steps - 2000 burn-in period avg:
Costs=1.6387380119085497 ± 0.0061 	 E[S]=1.2388 ± 0.0077 	 FR=0.9789 ± 0.0031 	 Total Eval Time=20.9898


(np.float64(1.6387380119085497),
 np.float64(0.006074134967664909),
 np.float64(0.9788825003815131),
 np.float64(0.0030580590190914967),
 np.float64(1.238755681818181),
 np.float64(0.00774879814394449))

In [ ]:
# Let us learn using a DQN policy
print("Learning a policy using DQN")
dqn = DDQN(env=inventory_rs)
ppo.learn(800000)
print("Finished Learning DQN policy, evaluating .......")
evaluate_policy(env=inventory_rs, policy=dqn, replication=8, processors=4)

Learning a policy using DQN
